# B1 mid-submission baseline — Qwen3-8B on Kaggle

Run this notebook on **Kaggle**, with **GPU T4 x2**, **Internet On**, and preferably **Persistence: Files only**. Phone verification may be required to enable these options. Import this `.ipynb` using Kaggle's **File → Import Notebook**.

This implements [mid_baselines_run.md](https://github.com/neill-nil/gst-scn-compliance-audit/blob/mid-baselines/docs/team_plan/mid_baselines_run.md): 54 reviewed gold cases × two views (`full_record`, `notice_only`), seed 1, temperature 0, thinking off, 16,384-token context, and two parallel requests. B1 uses local Ollama inference; no Gemini key is needed.

Run Cells 1–5 in order. Allow roughly 10 minutes for setup and 40–60 minutes for inference; actual time depends on Kaggle. The final artifact is **`/kaggle/working/b1_results.zip`**, containing all 108 predictions, test scores, usage logs, and run details.

The source checkout is pinned to commit `6c563663bbd486c5dbbff753e57797c911ade6b1` on `mid-baselines`, so a later branch update cannot silently change the experiment. The notebook uses the repository's prompts, model client, input builder, and scorer unchanged.


## Cell 1 — Start Ollama and download Qwen3-8B

Checks for two T4 GPUs, installs Ollama when needed, starts the server with the guide's settings, and waits until it responds. Model files and logs stay under `/kaggle/working`. Re-running this cell reuses a responding server and the downloaded model.


In [ ]:
import json
import os
from pathlib import Path
import re
import shlex
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

WORK = Path("/kaggle/working")
if not WORK.is_dir():
    raise RuntimeError("Run this notebook on Kaggle with GPU T4 x2 and Internet On.")

MODEL = "qwen3:8b"
SEED = 1
CONTEXT_LENGTH = 16384
PARALLEL_REQUESTS = 2
EXPECTED_CASES = 54
VIEWS = ("full_record", "notice_only")
REPO_URL = "https://github.com/neill-nil/gst-scn-compliance-audit.git"
REPO_BRANCH = "mid-baselines"
REPO_COMMIT = "6c563663bbd486c5dbbff753e57797c911ade6b1"
REPO = WORK / "repo"
OLLAMA_URL = "http://127.0.0.1:11434"
SERVER_LOG = WORK / "ollama.log"

def run(command, *, cwd=None, capture=False):
    print("$", shlex.join(map(str, command)), flush=True)
    result = subprocess.run(
        list(map(str, command)), cwd=cwd, check=True, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    return result.stdout if capture else result

def ollama_json(endpoint):
    with urllib.request.urlopen(OLLAMA_URL + endpoint, timeout=5) as response:
        return json.loads(response.read())

def server_ready():
    try:
        ollama_json("/api/tags")
        return True
    except (OSError, ValueError):
        return False

def tail(path, lines=8):
    path = Path(path)
    if not path.exists():
        return "(log not created yet)"
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU found. Select Accelerator: GPU T4 x2.")
gpu_report = run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture=True,
)
print(gpu_report)
gpu_lines = [line for line in gpu_report.splitlines() if line.strip()]
if len(gpu_lines) != 2 or any("T4" not in line for line in gpu_lines):
    raise RuntimeError("This experiment requires GPU T4 x2. Change Kaggle Session options.")

os.environ.update(
    OLLAMA_HOST="127.0.0.1:11434",
    OLLAMA_MODELS=str(WORK / "ollama_models"),
    OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH),
    OLLAMA_NUM_PARALLEL=str(PARALLEL_REQUESTS),
    OLLAMA_KEEP_ALIVE="-1",
    LOCAL_BASE_URL=OLLAMA_URL + "/v1",
    LOCAL_MODEL=MODEL,
    LOCAL_RPM="0",
)

if shutil.which("ollama") is None:
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "install", "-y", "zstd", "pciutils", "lshw"])
    run(["bash", "-o", "pipefail", "-c",
         "curl -fsSL https://ollama.com/install.sh | sh"])

if not server_ready():
    with SERVER_LOG.open("a", encoding="utf-8") as log:
        ollama_process = subprocess.Popen(
            ["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT,
            env=os.environ.copy(), start_new_session=True,
        )
    deadline = time.monotonic() + 90
    while not server_ready():
        if ollama_process.poll() is not None or time.monotonic() >= deadline:
            raise RuntimeError("Ollama did not become ready:\n" + tail(SERVER_LOG, 20))
        time.sleep(1)
else:
    print("Using the existing Ollama server. Cell 3 checks GPU placement and context.")

run(["ollama", "pull", MODEL])
model_info = next(
    (entry for entry in ollama_json("/api/tags")["models"]
     if entry.get("name") == MODEL or entry.get("model") == MODEL),
    None,
)
if model_info is None:
    raise RuntimeError(f"{MODEL} is not available after ollama pull.")
quantization = model_info.get("details", {}).get("quantization_level")
if quantization != "Q4_K_M":
    raise RuntimeError(f"The guide specifies Q4_K_M; downloaded model reports {quantization!r}.")
print("Ollama version:", ollama_json("/api/version").get("version"))
print("Model digest:", model_info.get("digest"))
print("Server ready. GPU placement is checked after the smoke test loads the model.")


## Cell 2 — Fetch the experiment code and build inputs

Clones `mid-baselines` into `/kaggle/working/repo` and selects the pinned commit. Existing checkouts are reused only when their commit matches and tracked files are clean; caches and outputs are preserved.

Builds all three views as in the guide, but runs inference only on `full_record` and `notice_only`. Checks that all 54 unique gold cases appear in the test split and that gold finding/outcome fields were removed by the input builder.


In [ ]:
if not (REPO / ".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f"{REPO} already exists and is not a Git checkout.")
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO])
    run(["git", "checkout", "--detach", REPO_COMMIT], cwd=REPO)

actual_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO, capture=True).strip()
if actual_commit != REPO_COMMIT:
    raise RuntimeError(
        f"Existing checkout is {actual_commit}, expected {REPO_COMMIT}. "
        "Use a fresh Kaggle session or a separate checkout to avoid mixing experiments."
    )
tracked_changes = run(
    ["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, capture=True,
)
if tracked_changes.strip():
    raise RuntimeError("Tracked source/data files were modified:\n" + tracked_changes)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
run([sys.executable, "-m", "pip", "-q", "install", "jsonschema"])

def load_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

gold_path = REPO / "data/gold/records.jsonl"
gold_records = load_jsonl(gold_path)
gold_cnrs = {record["cnr"] for record in gold_records}
if len(gold_records) != EXPECTED_CASES or len(gold_cnrs) != EXPECTED_CASES:
    raise RuntimeError(f"Expected {EXPECTED_CASES} unique gold cases, got {len(gold_records)} rows.")
test_cnrs = set((REPO / "data/splits/test.txt").read_text(encoding="utf-8").split())
if gold_cnrs != test_cnrs:
    raise RuntimeError("Gold case IDs do not match the test split. Check with Neill before running.")

run([sys.executable, "-m", "src.baselines.inputs",
     "--gold", "data/gold/records.jsonl", "--out", "data/interim/inputs"], cwd=REPO)

from src.baselines.inputs import FULL_RECORD_FIELDS, METADATA_FIELDS, NOTICE_ONLY_FIELDS
from eval.score import DEFECT_TYPES

for view in (*VIEWS, "raw_text"):
    rows = load_jsonl(REPO / f"data/interim/inputs/{view}.jsonl")
    if len(rows) != EXPECTED_CASES or {row["cnr"] for row in rows} != gold_cnrs:
        raise RuntimeError(f"{view}: case coverage differs from gold.")
    if any(row["view"] != view for row in rows):
        raise RuntimeError(f"{view}: an input has the wrong view.")
    for row in rows:
        record = row["record"]
        if view == "raw_text":
            allowed = {"paragraphs"}
        else:
            fields = FULL_RECORD_FIELDS if view == "full_record" else NOTICE_ONLY_FIELDS
            allowed = fields | METADATA_FIELDS
        if set(record) - allowed:
            raise RuntimeError(f"{view}: unexpected input fields: {set(record) - allowed}")
    print(f"Verified {view}: {len(rows)} unique cases.")

print("Inputs ready. The model receives only the repository input builder's fact views.")


## Cell 3 — Test one case and verify GPU inference

Runs the first `full_record` case using the exact model and seed used in the full run. Stops if the findings are empty, the model runs partly on CPU, or the context is wrong. Partial defect lists are reported explicitly; predictions are preserved exactly as the baseline produced them.

If this cell fails, keep the logs and send the error to Neill. The test call is cached and reused when Cell 4 reaches the same case.


In [ ]:
def validate_outputs(path, expected_cnrs, expected_view):
    rows = load_jsonl(path)
    ids = [row["cnr"] for row in rows]
    if len(ids) != len(expected_cnrs) or len(set(ids)) != len(ids) or set(ids) != set(expected_cnrs):
        raise RuntimeError(f"{path}: missing, duplicate, or unexpected case IDs.")
    incomplete = []
    for row in rows:
        if row.get("view") != expected_view or row.get("system") != "b1_plain":
            raise RuntimeError(f"{path}: unexpected system/view for {row['cnr']}.")
        findings = row.get("findings")
        if not isinstance(findings, list) or not findings:
            raise RuntimeError(
                f"{row['cnr']} ({expected_view}): findings are empty. "
                "Keep the logs/cache and send them to Neill before proceeding."
            )
        types = [finding["type"] for finding in findings]
        if any(finding.get("verdict") not in {"defect", "no_defect", "cant_tell", "contested"}
               for finding in findings):
            raise RuntimeError(f"{row['cnr']}: unknown verdict.")
        if set(types) != set(DEFECT_TYPES) or len(types) != len(DEFECT_TYPES):
            incomplete.append({
                "cnr": row["cnr"], "view": expected_view,
                "missing_types": sorted(set(DEFECT_TYPES) - set(types)),
                "duplicate_types": sorted({kind for kind in types if types.count(kind) > 1}),
            })
    return rows, incomplete

if any(process.poll() is None for process in globals().get("b1_processes", {}).values()):
    raise RuntimeError("Cell 4 workers are already running. Wait or interrupt Cell 4 first.")

SMOKE_PASSED = False
FULL_RUN_PASSED = False
smoke_input = Path("/tmp/b1_one.jsonl")
smoke_output = Path("/tmp/b1_one/outputs.jsonl")
first_input = load_jsonl(REPO / "data/interim/inputs/full_record.jsonl")[0]
smoke_input.write_text(json.dumps(first_input) + "\n", encoding="utf-8")
run([sys.executable, "-m", "src.baselines.b1_plain",
     "--input", smoke_input, "--out", smoke_output,
     "--model", MODEL, "--seed", SEED, "--run-id", "b1_smoke"], cwd=REPO)
smoke_rows, smoke_warnings = validate_outputs(
    smoke_output, {first_input["cnr"]}, "full_record",
)
print(json.dumps(smoke_rows[0], indent=2)[:1200])
if smoke_warnings:
    print("Warning: incomplete finding coverage:", json.dumps(smoke_warnings, indent=2))

placement = run(["ollama", "ps"], capture=True)
print(placement)
model_lines = [line for line in placement.splitlines() if MODEL in line]
if not model_lines or not re.search(r"\b100%\s+GPU\b", model_lines[0]):
    raise RuntimeError("qwen3:8b must show 100% GPU. Check Kaggle GPU T4 x2, then rerun Cell 1.")

running_model = next(
    (entry for entry in ollama_json("/api/ps").get("models", [])
     if entry.get("name") == MODEL or entry.get("model") == MODEL),
    None,
)
if running_model is None or running_model.get("context_length") != CONTEXT_LENGTH:
    raise RuntimeError(
        "The active context must be 16384. Restart the Kaggle session, "
        "then run Cell 1 before loading any model."
    )

SMOKE_PASSED = True
print(f"Smoke test passed: {len(smoke_rows[0]['findings'])} findings, 100% GPU, context 16384.")


## Cell 4 — Run both views in parallel

Starts one baseline worker per view and prints progress every minute. Cache entries and logged B1 calls provide progress estimates; exact case coverage is checked when the workers finish. Baseline output files are written at the end of each view.

**Resuming:** with Files persistence, run Cells 1, 2, 3, then 4 again. The repository cache reuses completed calls. Do not delete `runs/cache`. The smoke test overlaps one full-run input, so a fresh successful run normally needs **108 unique calls**, rather than 109. Actual calls can be higher after errors/retries.

Interrupting this cell stops its two workers and keeps completed cache entries. A five-minute pause in cache growth prints diagnostics; no predictions or prompts are changed.


In [ ]:
if not globals().get("SMOKE_PASSED", False):
    raise RuntimeError("Run Cell 3 successfully before starting the experiment.")
if not server_ready():
    raise RuntimeError("Ollama stopped. Rerun Cell 1, then Cell 3.")
if any(process.poll() is None for process in globals().get("b1_processes", {}).values()):
    raise RuntimeError("Baseline workers are already running; do not launch duplicates.")

FULL_RUN_PASSED = False
run_started_utc = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
b1_processes = {}
log_handles = {}
run_logs = {view: REPO / f"b1_{view}.log" for view in VIEWS}

def progress_counts():
    cache_dir = REPO / "runs/cache"
    cached = sum(1 for _ in cache_dir.glob("*.json")) if cache_dir.exists() else 0
    call_log = REPO / "runs/llm_calls.jsonl"
    logged = 0
    if call_log.exists():
        for line in call_log.read_text(encoding="utf-8").splitlines():
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue  # another worker may be in the middle of appending a line
            if entry.get("backend") == "local" and entry.get("tag") == "b1":
                logged += 1
    return cached, logged

try:
    for view in VIEWS:
        log_handles[view] = run_logs[view].open("a", encoding="utf-8")
        command = [
            sys.executable, "-u", "-m", "src.baselines.b1_plain",
            "--input", f"data/interim/inputs/{view}.jsonl",
            "--out", f"runs/b1_{view}/outputs.jsonl",
            "--model", MODEL, "--seed", str(SEED), "--run-id", f"b1_{view}",
        ]
        print("Starting", view, flush=True)
        b1_processes[view] = subprocess.Popen(
            command, cwd=REPO, env=os.environ.copy(),
            stdout=log_handles[view], stderr=subprocess.STDOUT,
        )

    last_count, _ = progress_counts()
    last_change = time.monotonic()
    next_report = time.monotonic()
    while any(process.poll() is None for process in b1_processes.values()):
        failures = {view: process.returncode for view, process in b1_processes.items()
                    if process.poll() not in (None, 0)}
        if failures:
            raise RuntimeError(
                f"Baseline worker failed: {failures}\n" +
                "\n".join(f"{view}:\n{tail(run_logs[view], 12)}" for view in VIEWS)
            )
        now = time.monotonic()
        if now >= next_report:
            cached, logged = progress_counts()
            states = {view: "running" if process.poll() is None else process.returncode
                      for view, process in b1_processes.items()}
            print(time.strftime("%H:%M:%S"), f"cache={cached} (~108 target), "
                  f"logged B1 calls={logged}; workers={states}", flush=True)
            if cached != last_count:
                last_count, last_change = cached, now
            elif now - last_change >= 300:
                print("No new cache entries in five minutes. Current logs:", flush=True)
                print("Ollama:\n" + tail(SERVER_LOG, 8), flush=True)
                for view in VIEWS:
                    print(f"{view}:\n{tail(run_logs[view], 5)}", flush=True)
                last_change = now
            next_report = now + 60
        time.sleep(5)

    exit_codes = {view: process.wait() for view, process in b1_processes.items()}
    print("Exit codes:", exit_codes)
    if any(exit_codes.values()):
        raise RuntimeError(
            "A worker failed:\n" +
            "\n".join(f"{view}:\n{tail(run_logs[view], 12)}" for view in VIEWS)
        )
except BaseException:
    # Stop only the two workers launched by this cell.
    for process in b1_processes.values():
        if process.poll() is None:
            process.terminate()
    for process in b1_processes.values():
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    raise
finally:
    for handle in log_handles.values():
        handle.close()

output_quality_warnings = []
for view in VIEWS:
    rows, warnings = validate_outputs(
        REPO / f"runs/b1_{view}/outputs.jsonl", gold_cnrs, view,
    )
    output_quality_warnings.extend(warnings)
    print(f"{view}: verified {len(rows)} outputs; {len(warnings)} incomplete finding lists.")
    print(tail(run_logs[view], 2))

FULL_RUN_PASSED = True
print("Both views finished. Proceed to Cell 5 to score and create the ZIP.")


## Cell 5 — Score and export `b1_results.zip`

Checks both views again, merges exactly 108 outputs, then runs the official scorer with **`--split test --no-unargued-as-rejected`**. Gold `not_decided` findings are excluded and unargued defects are not treated as rejected.

The ZIP contains `b1/outputs.jsonl`, `b1/score_test.json`, both per-view output folders, `llm_calls.jsonl`, logs, and `b1/experiment.json` (commit, settings, model digest, data hashes, and any incomplete finding lists). Caches and downloaded model weights are not included.

Download **`b1_results.zip`** from Kaggle's Output panel and send it, together with this cell's printed scores, to Neill. Then **Stop session** in Kaggle to release GPU quota.


In [ ]:
import hashlib
import platform
import zipfile

if any(process.poll() is None for process in globals().get("b1_processes", {}).values()):
    raise RuntimeError("Inference is still running. Wait for Cell 4 to finish.")
if not globals().get("FULL_RUN_PASSED", False):
    raise RuntimeError("Complete Cell 4 successfully before packaging.")

merged = []
quality_warnings = []
for view in VIEWS:
    rows, warnings = validate_outputs(
        REPO / f"runs/b1_{view}/outputs.jsonl", gold_cnrs, view,
    )
    merged.extend(rows)
    quality_warnings.extend(warnings)
if len(merged) != EXPECTED_CASES * len(VIEWS):
    raise RuntimeError("Expected exactly 108 outputs before scoring.")

combined_dir = REPO / "runs/b1"
combined_dir.mkdir(parents=True, exist_ok=True)
(combined_dir / "outputs.jsonl").write_text(
    "".join(json.dumps(row) + "\n" for row in merged), encoding="utf-8",
)
score_command = [
    sys.executable, "-m", "eval.score", "--gold", "data/gold/records.jsonl",
    "--run", "runs/b1", "--split", "test", "--no-unargued-as-rejected",
]
score_printout = run(score_command, cwd=REPO, capture=True)
print(score_printout)
(combined_dir / "score_printout.txt").write_text(score_printout, encoding="utf-8")
score_path = combined_dir / "score_test.json"
if not score_path.is_file():
    raise RuntimeError("The scorer did not create score_test.json.")
score = json.loads(score_path.read_text(encoding="utf-8"))
if score.get("n_gold_cases") != EXPECTED_CASES or set(score.get("views", {})) != set(VIEWS):
    raise RuntimeError("The score does not cover all 54 gold cases and both views.")

call_log = REPO / "runs/llm_calls.jsonl"
if not call_log.is_file():
    raise RuntimeError("Missing llm_calls.jsonl; run Cell 4 before packaging.")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

experiment = {
    "repo_url": REPO_URL, "source_branch": REPO_BRANCH, "repo_commit": actual_commit,
    "model": MODEL, "model_digest": model_info.get("digest"),
    "model_details": model_info.get("details"),
    "ollama_version": ollama_json("/api/version").get("version"),
    "gpu": gpu_report.strip().splitlines(),
    "python_version": platform.python_version(),
    "seed": SEED, "temperature": 0.0, "max_tokens": 4096,
    "thinking": "off (/no_think in the repository wrapper)",
    "context_length": CONTEXT_LENGTH, "parallel_requests": PARALLEL_REQUESTS,
    "views": list(VIEWS), "cases_per_view": EXPECTED_CASES, "outputs": len(merged),
    "started_utc": run_started_utc,
    "packaged_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "scoring": {"split": "test", "unargued_as_rejected": False},
    "sha256": {
        "gold": sha256(gold_path),
        "test_split": sha256(REPO / "data/splits/test.txt"),
        **{view + "_inputs": sha256(REPO / f"data/interim/inputs/{view}.jsonl")
           for view in VIEWS},
    },
    "quality_warnings": quality_warnings,
}
(combined_dir / "experiment.json").write_text(
    json.dumps(experiment, indent=2) + "\n", encoding="utf-8",
)

archive_path = WORK / "b1_results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ("b1", "b1_full_record", "b1_notice_only"):
        for path in sorted((REPO / "runs" / folder).rglob("*")):
            if path.is_file():
                archive.write(path, str(path.relative_to(REPO / "runs")))
    archive.write(call_log, "llm_calls.jsonl")
    for view, log_path in run_logs.items():
        archive.write(log_path, f"logs/b1_{view}.log")
    if SERVER_LOG.exists():
        archive.write(SERVER_LOG, "logs/ollama.log")

required_files = {
    "b1/outputs.jsonl", "b1/score_test.json", "b1/experiment.json",
    "b1_full_record/outputs.jsonl", "b1_notice_only/outputs.jsonl", "llm_calls.jsonl",
}
with zipfile.ZipFile(archive_path) as archive:
    if not required_files <= set(archive.namelist()):
        raise RuntimeError("Result ZIP is missing required files.")
    bad_member = archive.testzip()
    if bad_member:
        raise RuntimeError(f"ZIP integrity check failed: {bad_member}")

print(f"READY: {archive_path} ({archive_path.stat().st_size / 1024:.1f} KiB)")
print(f"Verified: {len(merged)} predictions, test scores, and usage log.")
if quality_warnings:
    print(f"Report to Neill: {len(quality_warnings)} outputs have incomplete finding lists; "
          "details are in b1/experiment.json. Predictions were kept as returned.")
from IPython.display import FileLink, display
display(FileLink(str(archive_path)))
print("Download b1_results.zip and copy the score printout, then Stop session in Kaggle.")


## Optional diagnostics

Run the cell below if setup or inference fails. Send its output to Neill. An idle Ollama server may show no loaded model until Cell 3 runs. To resume an interrupted run with persistence enabled, rerun Cells 1–4, then Cell 5.

| Symptom | Action |
|---|---|
| Internet/download failure | Enable Internet in Session options and verify your Kaggle account if prompted. |
| No GPU / partial CPU inference | Select GPU T4 x2, restart the session, rerun from Cell 1. |
| Ollama connection failure | Rerun Cell 1 and then the failed step. |
| Wrong context length | Restart the session and run Cell 1 before loading a model. |
| Empty findings | Keep the cache/logs and contact Neill. Do not edit predictions or change the prompt. |
| Existing checkout differs | Use a fresh session or separate checkout to preserve the pinned experiment. |
| No new cache entries for >5 minutes | Inspect logs below; long calls may still be running. |

References: [project guide](https://github.com/neill-nil/gst-scn-compliance-audit/blob/mid-baselines/docs/team_plan/mid_baselines_run.md), [Ollama GPU placement and parallelism](https://github.com/ollama/ollama/blob/main/docs/faq.mdx).


In [ ]:
print("GPU status:")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
print("\nLoaded Ollama models:")
if shutil.which("ollama"):
    subprocess.run(["ollama", "ps"], check=False)
for label, path in [
    ("Ollama", WORK / "ollama.log"),
    ("full_record", REPO / "b1_full_record.log"),
    ("notice_only", REPO / "b1_notice_only.log"),
]:
    print(f"\n{label} — last 10 lines:\n{tail(path, 10)}")
